## Disk spilling — when a stage runs out of memory

**Symptom in the UI:** stage detail shows a non-zero **Spill (Memory)** or **Spill (Disk)** column. Executors are writing intermediate state to disk because it didn't fit in RAM.

**Why spilling hurts:** spilled data has to be written to disk, then re-read, often de-serialised, and joined back in. That round trip is orders of magnitude slower than staying in memory — a spilling stage can be many times slower than the same stage that fits.

**Remedies:**

- **Increase `spark.sql.shuffle.partitions`** — more, smaller partitions each fit in memory. Usually the first and cleanest lever.
- **Increase executor memory** — but mind the trade-off: bigger executors mean fewer of them per node.
- **Filter / aggregate earlier** to shrink the per-partition payload.
- **Enable AQE** — coalesce partitions to the optimal size at runtime.

**The exam pattern:** *"a job is shuffling 200 GB across 200 partitions and tasks are spilling to disk."* The cleanest single-config answer is **raise `spark.sql.shuffle.partitions`** so each partition lands in the 100–200 MB range — about **1000–2000 partitions** for 200 GB. Note the direction: *more* partitions, not fewer. Fewer partitions makes each one bigger, which spills harder.
